In [ ]:
import glob
import numpy as np
import librosa

segment_dur = 2
directories = ['guitar', 'non_guitar']
def load_segments(directory):
    segments = []
    for path in glob.glob(f'{directory}/*.wav'):
        signal, sr = librosa.load(path, sr=None, mono=True)
        seg_len = sr * segment_dur
        audio_segments = librosa.util.frame(signal, frame_length=seg_len, hop_length=seg_len, axis=0)
        segments.append(audio_segments)
    return np.concatenate(segments)

guitar_segments = load_segments('guitar')
non_guitar_segments = load_segments('non_guitar')

/Applications/anaconda3/envs/ai-py311/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_chords(path):

    chords = []
    with open(path) as f:
        for line in f:      
            start_s, end_s, chord = line.strip().split('\t')
            chords.append(chord)
    return chords


guitar_chords = []
for i in range(6):
    guitar_chords += get_chords(f'guitar/guitar_annotation.lab')

non_guitar_chords = []
for i in range(10):
    non_guitar_chords += get_chords(f'non_guitar/non_guitar_annotation.lab')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sr = 44100
X_audio = np.concatenate([guitar_segments, non_guitar_segments])
chroma = librosa.feature.chroma_stft(y=X_audio, sr=sr, n_fft=8192, hop_length=4096)
X = np.concatenate([chroma.mean(axis=-1), chroma.std(axis=-1)], axis=1)

# Labels: drop the inversion (e.g. 'C:maj/3' -> 'C:maj') -> 84 classes (12 roots x 7 qualities)
y = np.array([c.split('/')[0] for c in guitar_chords + non_guitar_chords])
print(X.shape, y.shape, len(set(y)), 'classes')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
scaler = StandardScaler().fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)```

(7398, 24) (7398,) 84 classes


In [4]:
# sklearn: SVM (RBF kernel)
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

svm = SVC(C=10).fit(X_train, y_train)
print('SVM test accuracy:', accuracy_score(y_test, svm.predict(X_test)))

SVM test accuracy: 0.8324324324324325


In [ ]:
# PyTorch: small MLP on the same features
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder().fit(y)
Xtr = torch.tensor(X_train, dtype=torch.float32)
Xte = torch.tensor(X_test, dtype=torch.float32)
ytr = torch.tensor(le.transform(y_train))
yte = torch.tensor(le.transform(y_test))

torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(256, len(le.classes_)),
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 64):
        idx = perm[i:i + 64]
        optimizer.zero_grad()
        loss = loss_fn(model(Xtr[idx]), ytr[idx])
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 20 == 0:
        model.eval()
        with torch.no_grad():
            acc = (model(Xte).argmax(1) == yte).float().mean().item()
        print(f'epoch {epoch + 1}: loss {loss.item():.3f}, test acc {acc:.3f}')

## Why chroma, and doing it by hand

A chord is defined by its **set of pitch classes** (C:maj = {C, E, G}), not by octave or instrument.
Chroma takes the spectrum and folds every frequency onto one of 12 pitch classes, so a C in any octave adds to the "C" bin.
That makes it largely independent of timbre (guitar vs piano vs pad) and octave, which is why it's the classic chord feature.

**Recipe (numpy only):**
1. FFT each 2 s clip. The clip holds a single sustained chord, so one FFT over the whole clip works and gives 0.5 Hz resolution.
2. Convert each FFT bin's frequency to a MIDI note: `midi = 69 + 12*log2(f/440)`. Round it, and sum the energy per note. This is the **semitone spectrum**: 84 notes from C1 to B7.
3. Fold the semitone spectrum mod 12 to get the **chroma**.

**The catch for inversions:** `C:maj/3` (E in the bass) has the same pitch classes as `C:maj`, so the chroma is identical.
Folding throws away the octave, which is exactly where the bass note is. Features that keep some octave information:
- **bass chroma:** chroma where lower notes get a bigger weight
- **lowest note:** pitch class of the lowest note with significant energy (one-hot)
- **the full semitone spectrum:** keep all 84 notes and let the model work out the bass
- **raw audio:** let a CNN learn everything from the waveform (the hardest option)

In [ ]:
# Manual semitone spectrum + chroma (numpy only, no librosa)
n_samples = X_audio.shape[1]                       # 88200 = 2 s at 44.1 kHz
freqs = np.fft.rfftfreq(n_samples, d=1 / sr)       # frequency (Hz) of each FFT bin

# Map each FFT bin to its nearest MIDI note
midi = np.full(len(freqs), -1.0)
midi[1:] = 69 + 12 * np.log2(freqs[1:] / 440)      # skip the 0 Hz bin (log of 0)
note = np.round(midi).astype(int)

midi_lo, midi_hi = 24, 108                          # C1 .. B7 -> 84 notes (7 octaves)
n_notes = midi_hi - midi_lo
valid = (note >= midi_lo) & (note < midi_hi)
bin_to_note = np.zeros((len(freqs), n_notes), dtype=np.float32)   # one-hot matrix: FFT bin -> note
bin_to_note[np.where(valid)[0], note[valid] - midi_lo] = 1

window = np.hanning(n_samples).astype(np.float32)  # taper the edges to reduce spectral leakage
semi = []
for i in range(0, len(X_audio), 256):              # in batches to save memory
    power = np.abs(np.fft.rfft(X_audio[i:i + 256] * window, axis=1)) ** 2
    semi.append(power.astype(np.float32) @ bin_to_note)  # sum the power falling in each note
semi = np.concatenate(semi)                         # (n_clips, 84)

# Chroma = fold octaves: reshape to (n, 7 octaves, 12 pitch classes) and sum the octaves
# (midi_lo = 24 is a C, so column 0 = C, 1 = C#, ...)
chroma_manual = semi.reshape(-1, n_notes // 12, 12).sum(axis=1)
chroma_manual /= chroma_manual.max(axis=1, keepdims=True)
print(semi.shape, chroma_manual.shape)

In [ ]:
# Bass-aware features
rel = semi / semi.max(axis=1, keepdims=True)  # loudness-normalised semitone spectrum

# bass chroma: weight each note by exp(-note/12), so each octave up counts ~e times less
weights = np.exp(-np.arange(n_notes) / 12)
bass_chroma = (rel * weights).reshape(-1, n_notes // 12, 12).sum(axis=1)
bass_chroma /= bass_chroma.max(axis=1, keepdims=True)

# lowest note with at least 10% of the loudest note's energy -> one-hot pitch class
lowest = np.argmax(rel > 0.1, axis=1)
lowest_pc = np.eye(12)[(lowest + midi_lo) % 12]

# log-compressed full semitone spectrum (keeps octave info)
log_semi = np.log1p(100 * rel).astype(np.float32)

feature_sets = {
    'chroma':         chroma_manual,
    'chroma+bass':    np.hstack([chroma_manual, bass_chroma]),
    'chroma+lowest':  np.hstack([chroma_manual, lowest_pc]),
    'semitone spec':  log_semi,
    'all':            np.hstack([chroma_manual, bass_chroma, lowest_pc, log_semi]),
}

# Labels: full (288 classes, with inversions) and without inversions (84 classes)
y_full = np.array(guitar_chords + non_guitar_chords)
y_noinv = np.array([c.split('/')[0] for c in y_full])

# One shared split (by index) so every model is compared on the same test clips
train_idx, test_idx = train_test_split(np.arange(len(y_full)), test_size=0.2, stratify=y_full, random_state=0)

In [ ]:
# sklearn SVM on each feature set, with and without inversions
from sklearn.pipeline import make_pipeline

for name, F in feature_sets.items():
    for label_name, labels in [('no inv (84)', y_noinv), ('with inv (288)', y_full)]:
        clf = make_pipeline(StandardScaler(), SVC(C=10)).fit(F[train_idx], labels[train_idx])
        print(f'{name:15s} {label_name:15s} acc = {clf.score(F[test_idx], labels[test_idx]):.3f}')

In [ ]:
# PyTorch CNN on the semitone spectrum, 288 classes (inversions included)
# Conv1d slides along the pitch axis: the same interval pattern (e.g. a major third) looks the same
# wherever it occurs, and the layers still know absolute position, so they can find the bass.
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'

def train_torch(model, X, labels, epochs, bs=64, lr=1e-3):
    enc = LabelEncoder().fit(labels)
    Xt, Y = torch.tensor(X, dtype=torch.float32), torch.tensor(enc.transform(labels))
    torch.manual_seed(0)
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        perm = torch.tensor(train_idx)[torch.randperm(len(train_idx))]
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            opt.zero_grad()
            loss = loss_fn(model(Xt[b].to(device)), Y[b].to(device))
            loss.backward()
            opt.step()
        if (epoch + 1) % max(1, epochs // 5) == 0:
            model.eval()
            with torch.no_grad():
                pred = torch.cat([model(Xt[test_idx][i:i + 256].to(device)).argmax(1).cpu()
                                  for i in range(0, len(test_idx), 256)])
            print(f'epoch {epoch + 1}: loss {loss.item():.3f}, test acc {(pred == Y[test_idx]).float().mean():.3f}')
    return model

cnn = nn.Sequential(
    nn.Conv1d(1, 32, 13, padding=6), nn.ReLU(),   # kernel 13 = just over an octave
    nn.Conv1d(32, 64, 13, padding=6), nn.ReLU(),
    nn.Conv1d(64, 64, 13, padding=6), nn.ReLU(),
    nn.Flatten(), nn.Dropout(0.3),
    nn.Linear(64 * 84, 256), nn.ReLU(),
    nn.Linear(256, len(set(y_full))),
)
cnn = train_torch(cnn, log_semi[:, None, :], y_full, epochs=50)  # (n, 1 channel, 84 notes)

In [ ]:
# PyTorch 1D CNN on raw audio, 288 classes. No hand-made features: the first conv layer has to learn
# its own frequency filters. An SVM can't do this: 88200 numbers per clip, and shifting the waveform
# by one sample changes every value even though it's the same chord.
raw = X_audio[:, ::4].astype(np.float32)               # 44.1 -> 11.025 kHz (crude, some aliasing)
raw /= np.abs(raw).max(axis=1, keepdims=True) + 1e-8   # normalise loudness per clip

def conv_block(c_in, c_out, k, stride=1):
    return [nn.Conv1d(c_in, c_out, k, stride=stride, padding=k // 2), nn.BatchNorm1d(c_out), nn.ReLU()]

raw_cnn = nn.Sequential(
    *conv_block(1, 32, 256, stride=4), nn.MaxPool1d(4),   # wide first kernel ~ learned filterbank
    *conv_block(32, 64, 9), nn.MaxPool1d(4),
    *conv_block(64, 128, 9), nn.MaxPool1d(4),
    *conv_block(128, 128, 9),
    nn.AdaptiveAvgPool1d(1), nn.Flatten(),               # average over time
    nn.Linear(128, len(set(y_full))),
)
raw_cnn = train_torch(raw_cnn, raw[:, None, :], y_full, epochs=30)   # ~3 min on Apple GPU